In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ------------------------------------------------------------
# 0) Load tokenizer + model
# ------------------------------------------------------------
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()


def as_legacy_kv(past_key_values):
    """Per-layer (k, v) tuples — works with DynamicCache (new HF) or legacy tuples."""
    if past_key_values is None:
        return None
    if hasattr(past_key_values, "to_legacy_cache"):
        return past_key_values.to_legacy_cache()
    return past_key_values


print("Loaded model:", model_name)
print("pad_token:", tokenizer.pad_token, "pad_token_id:", tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token, "eos_token_id:", tokenizer.eos_token_id)


# ------------------------------------------------------------
# 1) Tokenizer basics: batching, padding, masks, PyTorch tensors
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("1) TOKENIZER BASICS")
print("=" * 70)

texts = [
    "The cat sat on the mat.",
    "Tokenization is not the same as words."
]

batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("Keys:", batch.keys())
for k, v in batch.items():
    print(f"\n{k}:")
    print("  shape:", tuple(v.shape))
    print("  dtype:", v.dtype)
    print(v)

print("\nInterpretation:")
print("- input_ids are integer token IDs")
print("- attention_mask has 1 for real tokens and 0 for padding")


# ------------------------------------------------------------
# 2) Show tokens and subword pieces
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("2) SUBWORD / BPE TOKENS")
print("=" * 70)

examples = [
    "cat",
    " cats",
    "Tokenization",
    " Tokenization",
    "unbelievable",
    "Greg",
    "greg",
    "xqzvplm",
]

for s in examples:
    ids = tokenizer(s, add_special_tokens=False)["input_ids"]
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"\nText: {repr(s)}")
    print("IDs:   ", ids)
    print("Tokens:", toks)


# ------------------------------------------------------------
# 3) Forward pass: logits
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("3) MODEL OUTPUTS: LOGITS")
print("=" * 70)

with torch.no_grad():
    outputs = model(**batch, use_cache=True)

logits = outputs.logits
cache = outputs.past_key_values

print("logits shape:", tuple(logits.shape))
print("Interpretation: (batch_size, seq_len, vocab_size)")
print("Number of layers in cache:", len(as_legacy_kv(cache)))

# Look at top next-token predictions for the first example
last_logits = logits[0, -1]   # logits at final position for example 0
topk = torch.topk(last_logits, k=10)

top_ids = topk.indices.tolist()
top_vals = topk.values.tolist()
top_tokens = tokenizer.convert_ids_to_tokens(top_ids)

print("\nTop-10 next-token candidates for example 0:")
for tok, tid, val in zip(top_tokens, top_ids, top_vals):
    print(f"  token={repr(tok):>20}   id={tid:<8}   logit={val:.3f}")


# ------------------------------------------------------------
# 4) Simple next-token decode
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("4) GREEDY NEXT-TOKEN DECODING")
print("=" * 70)

prompt = "The capital of California is"
enc = tokenizer(prompt, return_tensors="pt")

print("Prompt:", repr(prompt))
print("Prompt token IDs:", enc["input_ids"][0].tolist())
print("Prompt tokens:", tokenizer.convert_ids_to_tokens(enc["input_ids"][0]))

with torch.no_grad():
    out = model(**enc, use_cache=True)

next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
next_text = tokenizer.decode(next_id[0])

print("\nPredicted next token id:", next_id.item())
print("Predicted next token text:", repr(next_text))


# ------------------------------------------------------------
# 5) KV cache demo: inspect cache shapes
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("5) KV CACHE INSPECTION")
print("=" * 70)

layer0_k, layer0_v = as_legacy_kv(out.past_key_values)[0]
print("Layer 0 key shape:  ", tuple(layer0_k.shape))
print("Layer 0 value shape:", tuple(layer0_v.shape))
print("Typical meaning: (batch, num_heads, seq_len, head_dim)")

def cache_seq_len(past_key_values):
    k, v = as_legacy_kv(past_key_values)[0]
    return k.shape[-2]

print("Initial cache sequence length:", cache_seq_len(out.past_key_values))

# Feed ONLY the new token, together with the cache
with torch.no_grad():
    out2 = model(
        input_ids=next_id,
        past_key_values=out.past_key_values,
        use_cache=True
    )

print("Logits shape after cached step:", tuple(out2.logits.shape))
print("Cache sequence length after cached step:", cache_seq_len(out2.past_key_values))

next_id_2 = out2.logits[:, -1, :].argmax(dim=-1, keepdim=True)
generated_ids = torch.cat([enc["input_ids"], next_id, next_id_2], dim=1)

print("Decoded after 2 generated tokens:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))


# ------------------------------------------------------------
# 6) Side-by-side: generation WITHOUT cache
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("6) GENERATION WITHOUT CACHE")
print("=" * 70)

prompt = "The capital of California is"
enc = tokenizer(prompt, return_tensors="pt")

generated_ids = enc["input_ids"].clone()
generated_mask = enc["attention_mask"].clone()

print("Initial decoded text:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

for step in range(3):
    with torch.no_grad():
        out_full = model(
            input_ids=generated_ids,
            attention_mask=generated_mask,
            use_cache=False
        )

    print(f"\nStep {step + 1}:")
    print("  input length fed to model:", generated_ids.shape[1])

    new_id = out_full.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    print("  new token:", repr(tokenizer.decode(new_id[0])))

    generated_ids = torch.cat([generated_ids, new_id], dim=1)
    generated_mask = torch.cat(
        [generated_mask, torch.ones((generated_mask.shape[0], 1), dtype=generated_mask.dtype)],
        dim=1
    )

print("\nFinal decoded text (no cache):")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))


# ------------------------------------------------------------
# 7) Side-by-side: generation WITH cache
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("7) GENERATION WITH CACHE")
print("=" * 70)

prompt = "The capital of California is"
enc = tokenizer(prompt, return_tensors="pt")

generated_ids = enc["input_ids"].clone()

with torch.no_grad():
    out = model(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        use_cache=True
    )

cache = out.past_key_values
new_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)

print("Initial pass:")
print("  input length fed to model:", enc["input_ids"].shape[1])
print("  predicted token:", repr(tokenizer.decode(new_id[0])))
print("  cache seq_len:", cache_seq_len(cache))

generated_ids = torch.cat([generated_ids, new_id], dim=1)

for step in range(2):
    with torch.no_grad():
        out = model(
            input_ids=new_id,           # only one token
            past_key_values=cache,
            use_cache=True
        )

    cache = out.past_key_values
    print(f"\nCached step {step + 1}:")
    print("  input length fed to model:", new_id.shape[1])
    print("  cache seq_len:", cache_seq_len(cache))

    new_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    print("  predicted token:", repr(tokenizer.decode(new_id[0])))

    generated_ids = torch.cat([generated_ids, new_id], dim=1)

print("\nFinal decoded text (with cache):")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded model: Qwen/Qwen2.5-0.5B-Instruct
pad_token: <|endoftext|> pad_token_id: 151643
eos_token: <|im_end|> eos_token_id: 151645

1) TOKENIZER BASICS
Keys: KeysView({'input_ids': tensor([[   785,   8251,   7578,    389,    279,   5517,     13, 151643, 151643],
        [  3323,   2022,    374,    537,    279,   1852,    438,   4244,     13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])})

input_ids:
  shape: (2, 9)
  dtype: torch.int64
tensor([[   785,   8251,   7578,    389,    279,   5517,     13, 151643, 151643],
        [  3323,   2022,    374,    537,    279,   1852,    438,   4244,     13]])

attention_mask:
  shape: (2, 9)
  dtype: torch.int64
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])

Interpretation:
- input_ids are integer token IDs
- attention_mask has 1 for real tokens and 0 for padding

2) SUBWORD / BPE TOKENS

Text: 'cat'
IDs:    [4616]
Tokens: ['cat']

Text: ' cats'
IDs:    [19423]
Tokens: 

TypeError: 'DynamicCache' object is not subscriptable

In [ ]:
prompt = "The capital of California is"

enc = tokenizer(prompt, return_tensors="pt")
input_ids = enc["input_ids"]
attention_mask = enc["attention_mask"]

print("Prompt:")
print(prompt)
print()

tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
for i, (tid, tok) in enumerate(zip(input_ids[0].tolist(), tokens)):
    print(f"{i:2d}: id={tid:6d}  token={repr(tok)}")

In [ ]:
with torch.no_grad():
    out = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=True
    )

print("logits shape:", tuple(out.logits.shape))
print("number of layers in cache:", len(as_legacy_kv(out.past_key_values)))

k0, v0 = as_legacy_kv(out.past_key_values)[0]
print("Layer 0 key shape:  ", tuple(k0.shape))
print("Layer 0 value shape:", tuple(v0.shape))

In [ ]:
next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
next_token = tokenizer.decode(next_id[0])

print("Next token id:", next_id.item())
print("Next token text:", repr(next_token))

In [ ]:
with torch.no_grad():
    out_cached = model(
        input_ids=next_id,                 # only ONE new token
        past_key_values=out.past_key_values,
        use_cache=True
    )

print("cached-step logits shape:", tuple(out_cached.logits.shape))

k0_new, v0_new = as_legacy_kv(out_cached.past_key_values)[0]
print("Layer 0 key shape after cached step:  ", tuple(k0_new.shape))
print("Layer 0 value shape after cached step:", tuple(v0_new.shape))